# 03 – Replay monthly files
Simulace zdrojového systému, který doručuje jeden soubor na měsíc pro transakční tabulky.
Načte kompletní extrakty ze zdrojového volumu a zapíše měsíční CSV soubory do landing vrstvy,
kde je Auto Loader postupně inkrementálně nahrává.


In [0]:
import os
import re

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# --- Parameters ---------------------------------------------------------------
dbutils.widgets.text("catalog", "olist_dev", "Target catalog")
dbutils.widgets.text("from_month", "2016-09", "From month (yyyy-MM)")
dbutils.widgets.text("to_month", "2017-12", "To month (yyyy-MM)")

CATALOG = dbutils.widgets.get("catalog")
FROM_MONTH = dbutils.widgets.get("from_month")
TO_MONTH = dbutils.widgets.get("to_month")

# Fail fast on invalid input instead of silently writing nothing
MONTH_PATTERN = r"^\d{4}-(0[1-9]|1[0-2])$"
for name, value in {"from_month": FROM_MONTH, "to_month": TO_MONTH}.items():
    if not re.match(MONTH_PATTERN, value):
        raise ValueError(f"{name} must be in format yyyy-MM, got '{value}'")
if FROM_MONTH > TO_MONTH:
    raise ValueError(f"from_month ({FROM_MONTH}) must not be after to_month ({TO_MONTH})")

# --- Paths & options ----------------------------------------------------------
SOURCE = f"/Volumes/{CATALOG}/landing/source/olist"
LANDING = f"/Volumes/{CATALOG}/landing/raw/olist"
CSV_OPTIONS = {"header": "true", "multiLine": "true", "escape": '"'}

# Tables without a date column; they are split by the order_id of monthly orders
RELATED_TABLES = ["order_items", "order_payments", "order_reviews"]

In [0]:
def read_source(table: str) -> DataFrame:
    """Read the full extract of one table from the source volume (all columns as strings)."""
    return spark.read.options(**CSV_OPTIONS).csv(f"{SOURCE}/{table}/")


def write_single_csv(df: DataFrame, table: str, month: str) -> int:
    """Write a monthly slice as one CSV file into landing and return the row count.

    Uses pandas because Spark writes a folder of part files, not a single file.
    Example: /Volumes/olist_dev/landing/raw/olist/orders/orders_2017_01.csv
    """
    target_dir = f"{LANDING}/{table}"
    os.makedirs(target_dir, exist_ok=True)
    path = f"{target_dir}/{table}_{month.replace('-', '_')}.csv"

    pdf = df.toPandas()  # compute once; reuse for writing and counting
    pdf.to_csv(path, index=False)
    return len(pdf)

In [0]:
# 1) Read full extracts once.
#    Timestamps look like "2017-10-02 10:56:33" -> first 7 characters = "yyyy-MM"
orders = read_source("orders").withColumn("_month", F.substring("order_purchase_timestamp", 1, 7))
related = {table: read_source(table) for table in RELATED_TABLES}

# 2) Months that exist in the data within the requested range, oldest first
months = [
    row["_month"]
    for row in orders.where(F.col("_month").between(FROM_MONTH, TO_MONTH))
    .select("_month")
    .distinct()
    .orderBy("_month")
    .collect()
]
if not months:
    raise ValueError(f"No orders found between {FROM_MONTH} and {TO_MONTH}")

# 3) For each month: write orders and their related rows, one file per table
summary = []
for month in months:
    # Helper column _month must not end up in the exported file
    orders_month = orders.where(F.col("_month") == month).drop("_month")
    order_ids = orders_month.select("order_id")

    row = {"month": month, "orders": write_single_csv(orders_month, "orders", month)}

    # left_semi = keep rows of df that have a matching order_id, add no columns
    for table, df in related.items():
        month_df = df.join(order_ids, on="order_id", how="left_semi")
        row[table] = write_single_csv(month_df, table, month)

    summary.append(row)
    print(f"OK  {month}")

# 4) Overview of written files (rows per table and month)
display(spark.createDataFrame(pd.DataFrame(summary)))